# Part 3 — Few-Shot Adaptation Under Domain Gap

This notebook combines the current Part 3 experiment script and plotting script.
It uses the same setup style as Notebook 02: Colab-friendly bootstrapping, local `src/` imports, and automatic CUDA/MPS/CPU device selection.

**Workflow**

1. Configure dataset, few-shot grid, adaptation methods, and repeated seeds.
2. Load a fixed pretrained ViT backbone.
3. Build few-shot source splits and shifted target splits.
4. Train PEFT methods and save result files under `outputs/part3/`.
5. Plot accuracy, generalization/overfitting, training curves, and budget/shift summaries from those files.


## 0. Colab / Repository Setup

In [ ]:
import os
import sys

if "google.colab" in sys.modules or "COLAB_GPU" in os.environ:
    if not os.path.exists("haicon_peft_co"):
        os.system("git clone https://github.com/trofimova/haicon_peft_co.git")
    os.chdir("haicon_peft_co")
    os.system("pip install -q transformers peft torchvision medmnist matplotlib pandas tqdm")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())


## 1. Imports And Device

In [ ]:
from __future__ import annotations

import copy
import json
import random
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset, TensorDataset
from transformers import ViTModel

ROOT = Path.cwd()
if (ROOT / ".." / "src").exists():
    ROOT = (ROOT / "..").resolve()
elif (ROOT / "src").exists():
    ROOT = ROOT.resolve()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.methods.adapters import AdapterHeadClassifier
from src.methods.bitfit import BitFitClassifier
from src.methods.linear_probe import LinearProbeModel
from src.methods.lora import LoRAClassifier
from src.methods.partial_ft import PartialFineTuneClassifier
from src.methods.prompt_tuning import LegacyPromptTunedClassifier, PromptTunedClassifier
from src.data import make_dataset_loaders
from src.training import count_trainable_parameters, evaluate, freeze_module, train_model

print("repo root:", ROOT)


## 2. Configuration

Edit this cell for the live experiment. `REPEAT_SEEDS` controls repeated few-shot resampling and model initialization.

In [ ]:
# Backbone
MODEL_NAME = "WinKawaks/vit-small-patch16-224"
IMG_SIZE = 224

# Dataset
# Choose: "eurosat" | "medmnist" | "cifar10" | "synthetic"
# DATASET_NAME = "eurosat"
DATASET_NAME = "cifar10"
# DATASET_NAME = "medmnist"
MEDMNIST_NAME = "pathmnist"
DATA_ROOT = str(ROOT / "data")
DATA_SUBSET_FRACTION = None
NUM_WORKERS = 0

# Synthetic fallback task. Only used when DATASET_NAME = "synthetic".
N_CLASSES = 5
RAW_POOL_SIZE = 360
VAL_PER_CLASS = 10
EVAL_PER_CLASS = 10
BATCH_SIZE = 16
SEED = 42
# REPEAT_SEEDS = [42, 43, 44]
REPEAT_SEEDS = [42]

# Few-shot and domain-gap knobs
# SHOTS_PER_CLASS = [1, 4, 8, 16]
# SHOTS_PER_CLASS = [2, 4, 12]
SHOTS_PER_CLASS = [1, 2, 4]
DOMAIN_SHIFT_STRENGTH = 0.5
TARGET_ADAPTATION_SHOTS = 0

# Keep the first run small. Add "adapter", "bitfit", "visual_prompt",
# "legacy_visual_prompt", or "partial_ft" once the basic run works.
METHODS_TO_RUN = [
    "linear_probe",
    "lora",
    "partial_ft",
    # "visual_prompt",
]

# Method hyperparameters
LORA_RANK = 8
LORA_TARGET = ["query", "value"]
ADAPTER_BOTTLENECK = 32
PROMPT_SIZE = 8
PARTIAL_N_BLOCKS = 12

# Optimization
DEFAULT_EPOCHS = 6
EPOCHS_BY_SHOT = {}
# EPOCHS_BY_SHOT = {
#     1: 6,
#     4: 6,
#     8: 6,
#     16: 6,
# }
LR_BY_METHOD = {
    "linear_probe": 5e-3,
    "lora": 3e-3,
    "adapter": 3e-3,
    "bitfit": 1e-3,
    "visual_prompt": 3e-3,
    "legacy_visual_prompt": 3e-3,
    "partial_ft": 1e-4,
}
WEIGHT_DECAY = 1e-2

# Output
PARAMETER_BUDGET = 250_000
OUTPUT_ROOT = ROOT / "outputs" / "part3"


## 3. Backbone And Runtime Helpers

In [ ]:
class HFViTBackbone(nn.Module):
    """Thin wrapper around a HuggingFace ViTModel that returns the CLS token."""

    def __init__(self, model_name: str = MODEL_NAME):
        super().__init__()
        self.vit = ViTModel.from_pretrained(model_name)
        self.feature_dim = self.vit.config.hidden_size

    def forward(self, x):
        return self.vit(pixel_values=x).last_hidden_state[:, 0]


def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def encode_in_batches(model, x, device, batch_size=32):
    feats = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(x), batch_size):
            xb = x[start : start + batch_size].to(device)
            feats.append(model(xb).cpu())
    return torch.cat(feats, dim=0)


def apply_domain_shift(x, strength=0.35, seed=123):
    if strength <= 0:
        return x.clone()

    g = torch.Generator().manual_seed(seed)
    shifted = x.clone()

    scales = torch.tensor(
        [
            1.0 + 0.70 * strength,
            1.0 - 0.45 * strength,
            1.0 + 0.20 * strength,
        ]
    ).view(1, 3, 1, 1)
    bias = torch.tensor([0.12 * strength, -0.08 * strength, 0.04 * strength]).view(
        1, 3, 1, 1
    )
    shifted = shifted * scales + bias

    pixels = max(1, int(IMG_SIZE * 0.04 * strength))
    shifted = shifted.roll(shifts=pixels, dims=-1)

    noise = torch.randn(shifted.shape, generator=g) * (0.25 * strength)
    return shifted + noise


## 4. Synthetic Fallback Helpers

In [ ]:
def make_teacher_labels(features, n_classes, seed=42):
    """Create balanced labels from frozen-backbone feature scores.

    A plain random linear teacher can collapse when random-noise images map to a
    narrow feature region. We still use feature-space scores, but assign a fixed
    quota of high-scoring examples to each class so few-shot splits are usable.
    """
    g = torch.Generator().manual_seed(seed)
    features = features - features.mean(dim=0, keepdim=True)
    features = F.normalize(features, dim=-1)
    teacher = torch.randn(features.shape[1], n_classes, generator=g)
    teacher = F.normalize(teacher, dim=0)
    scores = features @ teacher

    n = features.shape[0]
    base_quota = n // n_classes
    quotas = [base_quota] * n_classes
    for c in range(n % n_classes):
        quotas[c] += 1

    labels = torch.full((n,), -1, dtype=torch.long)
    assigned = torch.zeros(n, dtype=torch.bool)
    class_order = torch.randperm(n_classes, generator=g).tolist()

    for c in class_order:
        ranked = torch.argsort(scores[:, c], descending=True)
        taken = 0
        for idx in ranked.tolist():
            if assigned[idx]:
                continue
            labels[idx] = c
            assigned[idx] = True
            taken += 1
            if taken == quotas[c]:
                break

    if (labels < 0).any():
        raise RuntimeError("Balanced synthetic label assignment left examples unlabeled.")
    return labels


def class_index_map(y):
    by_class = defaultdict(list)
    for idx, label in enumerate(y.tolist()):
        by_class[int(label)].append(idx)
    return by_class


def make_synthetic_domains(backbone, device):
    g = torch.Generator().manual_seed(SEED)
    x_source = torch.randn(RAW_POOL_SIZE, 3, IMG_SIZE, IMG_SIZE, generator=g)
    features = encode_in_batches(backbone, x_source, device, batch_size=BATCH_SIZE)
    y_all = make_teacher_labels(features, N_CLASSES, seed=SEED)
    x_target = apply_domain_shift(
        x_source, strength=DOMAIN_SHIFT_STRENGTH, seed=SEED + 1
    )

    by_class = class_index_map(y_all)
    required_per_class = max(SHOTS_PER_CLASS) + TARGET_ADAPTATION_SHOTS + VAL_PER_CLASS
    counts = {c: len(by_class[c]) for c in range(N_CLASSES)}

    print("Examples per class:", counts)
    print("Required per class:", required_per_class)

    missing = [c for c, n in counts.items() if n < required_per_class]
    if missing:
        raise ValueError(
            f"Not enough examples for classes {missing}. Increase RAW_POOL_SIZE "
            "or reduce shots/validation size."
        )

    return x_source, x_target, y_all, by_class


## 5. Dataset And Few-Shot Split Helpers

For real datasets, `source` means the clean validation subset and `target` means the same subset after the controlled distortion from `DOMAIN_SHIFT_STRENGTH`.

In [ ]:
def load_experiment_data(backbone, device):
    if DATASET_NAME == "synthetic":
        x_source, x_target, y_all, by_class = make_synthetic_domains(backbone, device)
        return {
            "kind": "synthetic",
            "name": "synthetic",
            "num_classes": N_CLASSES,
            "x_source": x_source,
            "x_target": x_target,
            "y_all": y_all,
            "by_class": by_class,
        }

    bundle = make_dataset_loaders(
        DATASET_NAME,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        root=DATA_ROOT,
        image_size=IMG_SIZE,
        subset_fraction=DATA_SUBSET_FRACTION,
        seed=SEED,
        medmnist_name=MEDMNIST_NAME,
    )
    by_class = class_index_map_from_dataset(bundle.train_dataset, bundle.num_classes)
    required_per_class = max(SHOTS_PER_CLASS) + TARGET_ADAPTATION_SHOTS
    counts = {c: len(by_class[c]) for c in range(bundle.num_classes)}

    print("Examples per class in train split:", counts)
    print("Required per class:", required_per_class)

    missing = [c for c, n in counts.items() if n < required_per_class]
    if missing:
        raise ValueError(
            f"Not enough training examples for classes {missing}. Increase "
            "DATA_SUBSET_FRACTION, reduce shots, or choose another dataset."
        )

    return {
        "kind": "dataset",
        "name": DATASET_NAME,
        "num_classes": bundle.num_classes,
        "train_dataset": bundle.train_dataset,
        "val_dataset": bundle.val_dataset,
        "test_dataset": bundle.test_dataset,
        "by_class": by_class,
    }


class ShiftedDataset(Dataset):
    """Dataset wrapper that applies the same artificial domain shift as target data."""

    def __init__(self, dataset: Dataset, strength: float, seed: int):
        self.dataset = dataset
        self.strength = strength
        self.seed = seed

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        x_shifted = apply_domain_shift(
            x.unsqueeze(0), strength=self.strength, seed=self.seed + idx
        ).squeeze(0)
        return x_shifted, y


def get_label(dataset: Dataset, idx: int) -> int:
    """Read a class label without assuming one particular torchvision dataset type."""
    if isinstance(dataset, Subset):
        return get_label(dataset.dataset, int(dataset.indices[idx]))

    if hasattr(dataset, "targets"):
        return int(dataset.targets[idx])

    if hasattr(dataset, "labels"):
        label = torch.as_tensor(dataset.labels[idx]).reshape(-1)[0]
        return int(label.item())

    if hasattr(dataset, "dataset"):
        wrapped = dataset.dataset
        if hasattr(wrapped, "labels"):
            label = torch.as_tensor(wrapped.labels[idx]).reshape(-1)[0]
            return int(label.item())

    _, label = dataset[idx]
    return int(torch.as_tensor(label).reshape(-1)[0].item())


def class_index_map_from_dataset(dataset: Dataset, num_classes: int):
    by_class = defaultdict(list)
    for idx in range(len(dataset)):
        label = get_label(dataset, idx)
        if 0 <= label < num_classes:
            by_class[label].append(idx)
    return by_class


def take_balanced_indices(by_class, shots_per_class, num_classes, offset=0, seed=None):
    chosen = []
    for c in range(num_classes):
        class_indices = by_class[c]
        if seed is None:
            ordered = class_indices
        else:
            g = torch.Generator().manual_seed(seed + c)
            perm = torch.randperm(len(class_indices), generator=g).tolist()
            ordered = [class_indices[i] for i in perm]
        chosen.extend(ordered[offset : offset + shots_per_class])
    return chosen


def make_synthetic_few_shot_loaders(
    x_source, x_target, y_all, by_class, shots_per_class, repeat_seed
):
    train_idx = take_balanced_indices(
        by_class, shots_per_class, N_CLASSES, offset=0, seed=repeat_seed
    )
    next_offset = shots_per_class

    x_train = x_source[train_idx]
    y_train = y_all[train_idx]

    if TARGET_ADAPTATION_SHOTS > 0:
        target_train_idx = take_balanced_indices(
            by_class,
            TARGET_ADAPTATION_SHOTS,
            N_CLASSES,
            offset=next_offset,
            seed=repeat_seed,
        )
        x_train = torch.cat([x_train, x_target[target_train_idx]], dim=0)
        y_train = torch.cat([y_train, y_all[target_train_idx]], dim=0)
        next_offset += TARGET_ADAPTATION_SHOTS

    val_idx = take_balanced_indices(
        by_class, VAL_PER_CLASS, N_CLASSES, offset=next_offset, seed=repeat_seed
    )
    train = TensorDataset(x_train, y_train)
    source_val = TensorDataset(x_source[val_idx], y_all[val_idx])
    target_val = TensorDataset(x_target[val_idx], y_all[val_idx])

    train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
    source_loader = DataLoader(source_val, batch_size=BATCH_SIZE, shuffle=False)
    target_loader = DataLoader(target_val, batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, source_loader, target_loader


def make_dataset_few_shot_loaders(data, shots_per_class, repeat_seed):
    train_idx = take_balanced_indices(
        data["by_class"],
        shots_per_class,
        data["num_classes"],
        offset=0,
        seed=repeat_seed,
    )
    next_offset = shots_per_class

    train_dataset = Subset(data["train_dataset"], train_idx)

    if TARGET_ADAPTATION_SHOTS > 0:
        target_train_idx = take_balanced_indices(
            data["by_class"],
            TARGET_ADAPTATION_SHOTS,
            data["num_classes"],
            offset=next_offset,
            seed=repeat_seed,
        )
        target_train = ShiftedDataset(
            Subset(data["train_dataset"], target_train_idx),
            strength=DOMAIN_SHIFT_STRENGTH,
            seed=repeat_seed + 10_000,
        )
        train_dataset = ConcatDataset([train_dataset, target_train])

    eval_by_class = class_index_map_from_dataset(data["val_dataset"], data["num_classes"])
    eval_idx = take_balanced_indices(
        eval_by_class,
        min(EVAL_PER_CLASS, min(len(v) for v in eval_by_class.values())),
        data["num_classes"],
        offset=0,
        seed=repeat_seed,
    )
    source_eval = Subset(data["val_dataset"], eval_idx)
    target_eval = ShiftedDataset(
        source_eval, strength=DOMAIN_SHIFT_STRENGTH, seed=repeat_seed + 20_000
    )

    train_generator = torch.Generator().manual_seed(repeat_seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        generator=train_generator,
    )
    source_loader = DataLoader(
        source_eval, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
    )
    target_loader = DataLoader(
        target_eval, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
    )
    return train_loader, source_loader, target_loader


def make_few_shot_loaders(data, shots_per_class, repeat_seed):
    if data["kind"] == "synthetic":
        return make_synthetic_few_shot_loaders(
            data["x_source"],
            data["x_target"],
            data["y_all"],
            data["by_class"],
            shots_per_class,
            repeat_seed,
        )
    return make_dataset_few_shot_loaders(data, shots_per_class, repeat_seed)


## 6. Model Factory, Training, And Saving

In [ ]:
def build_model(
    method: str, backbone: nn.Module, feature_dim: int, num_classes: int
) -> nn.Module:
    bb = copy.deepcopy(backbone)

    if method == "linear_probe":
        return LinearProbeModel(bb, feature_dim, num_classes)

    if method == "bitfit":
        return BitFitClassifier(bb, feature_dim, num_classes)

    if method == "visual_prompt":
        return PromptTunedClassifier(bb, feature_dim, num_classes, prompt_size=PROMPT_SIZE)

    if method == "legacy_visual_prompt":
        return LegacyPromptTunedClassifier(
            bb, feature_dim, num_classes, prompt_size=PROMPT_SIZE
        )

    if method == "lora":
        return LoRAClassifier(
            bb,
            feature_dim,
            num_classes,
            target_modules=LORA_TARGET,
            rank=LORA_RANK,
        )

    if method == "adapter":
        return AdapterHeadClassifier(
            bb, feature_dim, num_classes, bottleneck_dim=ADAPTER_BOTTLENECK
        )

    if method == "partial_ft":
        blocks = list(bb.vit.encoder.layer)
        return PartialFineTuneClassifier(
            bb,
            feature_dim,
            num_classes,
            modules_to_unfreeze=blocks[-PARTIAL_N_BLOCKS:],
        )

    raise ValueError(f"Unknown method: {method}")


def optimizer_for(method, model):
    lr = LR_BY_METHOD.get(method, 3e-3)
    trainable = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(trainable, lr=lr, weight_decay=WEIGHT_DECAY)


def adapted_features(model, x):
    if hasattr(model, "forward_features"):
        return model.forward_features(x)

    if hasattr(model, "prompt"):
        x = model.prompt(x)
    feats = model.backbone(x)
    if hasattr(model, "adapter"):
        feats = model.adapter(feats)
    return feats


@torch.no_grad()
def mean_feature_shift(method, model, frozen_backbone, loader, device):
    if method == "linear_probe":
        return 0.0

    model.eval()
    frozen_backbone.eval()
    distances = []
    for x, _ in loader:
        x = x.to(device)
        base = frozen_backbone(x)
        adapted = adapted_features(model, x)
        distance = 1.0 - F.cosine_similarity(base, adapted, dim=-1)
        distances.append(distance.cpu())
    return torch.cat(distances).mean().item()


def run_experiment(
    method,
    shots_per_class,
    repeat_seed,
    backbone,
    feature_dim,
    data,
    device,
):
    train_loader, source_loader, target_loader = make_few_shot_loaders(
        data, shots_per_class, repeat_seed
    )
    set_seed(repeat_seed)
    model = build_model(method, backbone, feature_dim, data["num_classes"]).to(device)
    trainable = count_trainable_parameters(model)
    total = sum(p.numel() for p in model.parameters())
    opt = optimizer_for(method, model)
    epochs = EPOCHS_BY_SHOT.get(shots_per_class, DEFAULT_EPOCHS)
    
    # if method == "partial_ft":
    #     epochs = 2 * epochs 

    print(
        f"\n{method} | shots/class={shots_per_class} | repeat_seed={repeat_seed} | "
        f"trainable={trainable:,} | epochs={epochs}"
    )
    history = train_model(model, train_loader, source_loader, opt, epochs=epochs, device=device)
    train_metrics = {
        "loss": history.train_loss[-1],
        "acc": history.train_acc[-1],
    }
    print("  evaluating source split ...", flush=True)
    source_metrics = evaluate(model, source_loader, device=device)
    print("  evaluating shifted target split ...", flush=True)
    target_metrics = evaluate(model, target_loader, device=device)
    print("  measuring feature shift ...", flush=True)
    feature_shift = mean_feature_shift(method, model, backbone, target_loader, device)

    result = {
        "method": method,
        "shots_per_class": shots_per_class,
        "repeat_seed": repeat_seed,
        "train_examples": len(train_loader.dataset),
        "epochs": epochs,
        "trainable_params": trainable,
        "total_params": total,
        "trainable_pct": 100 * trainable / total,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "source_loss": source_metrics["loss"],
        "source_acc": source_metrics["acc"],
        "target_loss": target_metrics["loss"],
        "target_acc": target_metrics["acc"],
        "generalization_gap": train_metrics["acc"] - source_metrics["acc"],
        "domain_gap": source_metrics["acc"] - target_metrics["acc"],
        "feature_shift": feature_shift,
        "history": history,
    }

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


def experiment_config(device: str, data: dict, feature_dim: int) -> dict:
    return {
        "model_name": MODEL_NAME,
        "feature_dim": feature_dim,
        "device": device,
        "dataset_name": DATASET_NAME,
        "medmnist_name": MEDMNIST_NAME if DATASET_NAME == "medmnist" else None,
        "data_subset_fraction": DATA_SUBSET_FRACTION,
        "num_classes": data["num_classes"],
        "image_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "seed": SEED,
        "repeat_seeds": REPEAT_SEEDS,
        "shots_per_class": SHOTS_PER_CLASS,
        "eval_per_class": EVAL_PER_CLASS,
        "domain_shift_strength": DOMAIN_SHIFT_STRENGTH,
        "target_adaptation_shots": TARGET_ADAPTATION_SHOTS,
        "methods_to_run": METHODS_TO_RUN,
        "lora_rank": LORA_RANK,
        "lora_target": LORA_TARGET,
        "adapter_bottleneck": ADAPTER_BOTTLENECK,
        "prompt_size": PROMPT_SIZE,
        "partial_n_blocks": PARTIAL_N_BLOCKS,
        "default_epochs": DEFAULT_EPOCHS,
        "epochs_by_shot": EPOCHS_BY_SHOT,
        "lr_by_method": LR_BY_METHOD,
        "weight_decay": WEIGHT_DECAY,
        "parameter_budget": PARAMETER_BUDGET,
    }


def make_output_dir() -> Path:
    dataset_tag = DATASET_NAME if DATASET_NAME != "medmnist" else f"medmnist-{MEDMNIST_NAME}"
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    run_dir = OUTPUT_ROOT / f"{timestamp}_{dataset_tag}"
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir


def history_to_rows(results: list[dict]) -> list[dict]:
    rows = []
    for result in results:
        history = result["history"]
        for epoch, (train_loss, train_acc, val_loss, val_acc) in enumerate(
            zip(history.train_loss, history.train_acc, history.val_loss, history.val_acc),
            start=1,
        ):
            rows.append(
                {
                    "method": result["method"],
                    "shots_per_class": result["shots_per_class"],
                    "repeat_seed": result["repeat_seed"],
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "train_acc": train_acc,
                    "val_loss": val_loss,
                    "val_acc": val_acc,
                }
            )
    return rows


def save_results(results: list[dict], config: dict, run_dir: Path) -> pd.DataFrame:
    df = pd.DataFrame([{k: v for k, v in row.items() if k != "history"} for row in results])
    history_df = pd.DataFrame(history_to_rows(results))

    df.to_csv(run_dir / "results.csv", index=False)
    df.to_json(run_dir / "results.json", orient="records", indent=2)
    history_df.to_csv(run_dir / "history.csv", index=False)
    with (run_dir / "config.json").open("w") as f:
        json.dump(config, f, indent=2)

    print(f"\nSaved result files to {run_dir}")
    return df


## 7. Plotting Helpers

These functions are copied from `notebooks/plot_part3_results.py`, so the notebook can plot from saved result files without leaving the notebook.

In [ ]:
from matplotlib.lines import Line2D

DEFAULT_OUTPUT_ROOT = OUTPUT_ROOT
LINESTYLES = ["-", "--", ":", "-."]


def latest_run_dir(output_root: Path = DEFAULT_OUTPUT_ROOT) -> Path:
    candidates = [p for p in output_root.iterdir() if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f"No result directories found under {output_root}")
    return max(candidates, key=lambda p: p.stat().st_mtime)


def load_run(run_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame | None, dict]:
    results_path = run_dir / "results.csv"
    history_path = run_dir / "history.csv"
    config_path = run_dir / "config.json"

    if not results_path.exists():
        raise FileNotFoundError(f"Missing {results_path}")

    df = pd.read_csv(results_path)
    history_df = pd.read_csv(history_path) if history_path.exists() else None
    config = json.loads(config_path.read_text()) if config_path.exists() else {}
    return df, history_df, config


def plot_accuracy_grid(df: pd.DataFrame, config: dict):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    plot_df = df
    if "repeat_seed" in df.columns:
        metric_cols = ["source_acc", "target_acc", "domain_gap"]
        plot_df = df.groupby(["method", "shots_per_class"], as_index=False)[metric_cols].mean()

    for method, group in plot_df.groupby("method"):
        group = group.sort_values("shots_per_class")
        axes[0].plot(group["shots_per_class"], group["source_acc"], marker="o", label=method)
        axes[1].plot(group["shots_per_class"], group["target_acc"], marker="o", label=method)
        axes[2].plot(group["shots_per_class"], group["domain_gap"], marker="o", label=method)

    for ax in axes:
        ax.set_xlabel("Shots per class")
        ax.set_xscale("log")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    axes[0].set_title("In-domain accuracy")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_ylim(0, 1)

    axes[1].set_title("Shifted-domain accuracy")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_ylim(0, 1)

    axes[2].set_title("Source-target gap")
    axes[2].set_ylabel("Accuracy gap")

    dataset = config.get("dataset_name", "dataset")
    shift = config.get("domain_shift_strength", "?")
    fig.suptitle(f"{dataset} | domain shift={shift}", fontsize=12)
    fig.tight_layout()
    return fig


def plot_budget_and_shift(df: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    plot_df = df
    if "repeat_seed" in df.columns:
        metric_cols = [
            "trainable_params",
            "target_acc",
            "feature_shift",
        ]
        plot_df = df.groupby(["method", "shots_per_class"], as_index=False)[metric_cols].mean()

    scatter = axes[0].scatter(
        plot_df["trainable_params"],
        plot_df["target_acc"],
        c=plot_df["shots_per_class"],
        s=90,
        cmap="viridis",
        zorder=3,
    )
    for _, row in plot_df.iterrows():
        axes[0].annotate(
            row["method"],
            (row["trainable_params"], row["target_acc"]),
            textcoords="offset points",
            xytext=(5, 4),
            fontsize=8,
        )
    axes[0].set_xscale("log")
    axes[0].set_xlabel("Trainable parameters")
    axes[0].set_ylabel("Target accuracy")
    axes[0].set_title("Quality vs parameter budget")
    axes[0].grid(True, alpha=0.3)
    fig.colorbar(scatter, ax=axes[0], label="shots/class")

    axes[1].scatter(plot_df["feature_shift"], plot_df["target_acc"], s=90, zorder=3)
    for _, row in plot_df.iterrows():
        label = f'{row["method"]} ({row["shots_per_class"]})'
        axes[1].annotate(
            label,
            (row["feature_shift"], row["target_acc"]),
            textcoords="offset points",
            xytext=(5, 4),
            fontsize=8,
        )
    axes[1].set_xlabel("Mean feature cosine distance from frozen backbone")
    axes[1].set_ylabel("Target accuracy")
    axes[1].set_title("Did moving features help?")
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    return fig


def method_colors(methods):
    color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    return {method: color_cycle[i % len(color_cycle)] for i, method in enumerate(methods)}


def shot_linestyles(shots):
    ordered = sorted(shots, reverse=True)
    return {shot: LINESTYLES[i % len(LINESTYLES)] for i, shot in enumerate(ordered)}


def aggregate_results(df: pd.DataFrame, metrics: list[str]) -> pd.DataFrame:
    present_metrics = [metric for metric in metrics if metric in df.columns]
    group_cols = ["method", "shots_per_class"]
    if "repeat_seed" not in df.columns or df["repeat_seed"].nunique() <= 1:
        return df[group_cols + present_metrics].copy()

    mean_df = df.groupby(group_cols, as_index=False)[present_metrics].mean()
    std_df = df.groupby(group_cols, as_index=False)[present_metrics].std()
    std_df = std_df.rename(columns={metric: f"{metric}_std" for metric in present_metrics})
    return mean_df.merge(std_df, on=group_cols, how="left")


def plot_metric_lines(ax, df: pd.DataFrame, metric: str, colors: dict, ylabel: str, title: str):
    for method, group in df.groupby("method"):
        group = group.sort_values("shots_per_class")
        yerr = group[f"{metric}_std"] if f"{metric}_std" in group.columns else None
        ax.errorbar(
            group["shots_per_class"],
            group[metric],
            yerr=yerr,
            marker="o",
            capsize=3 if yerr is not None else 0,
            color=colors[method],
            label=method,
        )
    ax.set_xscale("log")
    ax.set_xlabel("Shots per class")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)


def plot_generalization_summary(df: pd.DataFrame):
    required = {
        "train_acc",
        "source_acc",
        "target_acc",
        "generalization_gap",
        "domain_gap",
        "train_loss",
        "source_loss",
        "target_loss",
    }
    if not required.issubset(df.columns):
        missing = ", ".join(sorted(required - set(df.columns)))
        raise ValueError(f"Cannot plot generalization summary; missing columns: {missing}")

    metrics = sorted(required)
    plot_df = aggregate_results(df, metrics)
    methods = sorted(plot_df["method"].unique())
    colors = method_colors(methods)

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))

    for method, group in plot_df.groupby("method"):
        group = group.sort_values("shots_per_class")
        color = colors[method]
        for metric, linestyle, label in [
            ("train_acc", "-", "train"),
            ("source_acc", "--", "source val"),
            ("target_acc", ":", "target val"),
        ]:
            yerr = group[f"{metric}_std"] if f"{metric}_std" in group.columns else None
            axes[0, 0].errorbar(
                group["shots_per_class"],
                group[metric],
                yerr=yerr,
                marker="o",
                linestyle=linestyle,
                capsize=3 if yerr is not None else 0,
                color=color,
                label=f"{method} {label}",
            )

        for metric, linestyle, label in [
            ("train_loss", "-", "train"),
            ("source_loss", "--", "source val"),
            ("target_loss", ":", "target val"),
        ]:
            yerr = group[f"{metric}_std"] if f"{metric}_std" in group.columns else None
            axes[1, 0].errorbar(
                group["shots_per_class"],
                group[metric],
                yerr=yerr,
                marker="o",
                linestyle=linestyle,
                capsize=3 if yerr is not None else 0,
                color=color,
                label=f"{method} {label}",
            )

    axes[0, 0].set_title("Accuracy: train vs source vs target")
    axes[0, 0].set_ylabel("Accuracy")
    axes[0, 0].set_ylim(0, 1)
    axes[1, 0].set_title("Loss: train vs source vs target")
    axes[1, 0].set_ylabel("Loss")

    plot_metric_lines(
        axes[0, 1], plot_df, "generalization_gap", colors,
        "Train acc - source acc", "Generalization gap",
    )
    plot_metric_lines(
        axes[0, 2], plot_df, "domain_gap", colors,
        "Source acc - target acc", "Domain gap",
    )
    plot_metric_lines(
        axes[1, 1], plot_df, "source_loss", colors,
        "Source validation loss", "Source validation loss",
    )
    plot_metric_lines(
        axes[1, 2], plot_df, "target_loss", colors,
        "Target validation loss", "Target validation loss",
    )

    for ax in axes.ravel():
        ax.set_xscale("log")
        ax.set_xlabel("Shots per class")
        ax.grid(True, alpha=0.3)

    method_handles = [
        Line2D([0], [0], color=colors[method], lw=2, label=method)
        for method in methods
    ]
    split_handles = [
        Line2D([0], [0], color="black", lw=2, linestyle="-", label="train"),
        Line2D([0], [0], color="black", lw=2, linestyle="--", label="source val"),
        Line2D([0], [0], color="black", lw=2, linestyle=":", label="target val"),
    ]
    method_legend = axes[0, 0].legend(
        handles=method_handles, title="Method", fontsize=8, title_fontsize=9,
        loc="lower right", handlelength=3.2,
    )
    axes[0, 0].add_artist(method_legend)
    axes[1, 0].legend(
        handles=split_handles, title="Split", fontsize=8, title_fontsize=9,
        loc="upper right", handlelength=4.0,
    )

    fig.tight_layout()
    return fig


def plot_training_curves(history_df: pd.DataFrame):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
    methods = sorted(history_df["method"].unique())
    shots_values = sorted(history_df["shots_per_class"].unique(), reverse=True)
    colors = method_colors(methods)
    linestyles = shot_linestyles(shots_values)
    has_train_acc = "train_acc" in history_df.columns

    group_cols = ["method", "shots_per_class", "epoch"]
    mean_cols = ["train_loss", "val_loss", "val_acc"]
    if has_train_acc:
        mean_cols.append("train_acc")
    mean_df = history_df.groupby(group_cols, as_index=False)[mean_cols].mean()

    repeat_groups = []
    if "repeat_seed" in history_df.columns and history_df["repeat_seed"].nunique() > 1:
        repeat_groups = list(history_df.groupby(["method", "shots_per_class", "repeat_seed"]))

    for (method, shots, _repeat_seed), group in repeat_groups:
        group = group.sort_values("epoch")
        style = {
            "color": colors[method],
            "linestyle": linestyles[shots],
            "linewidth": 0.8,
            "alpha": 0.18,
        }
        axes[0, 0].plot(group["epoch"], group["train_loss"], **style)
        axes[0, 1].plot(group["epoch"], group["val_loss"], **style)
        if has_train_acc:
            axes[1, 0].plot(group["epoch"], group["train_acc"], **style)
        axes[1, 1].plot(group["epoch"], group["val_acc"], **style)

    for (method, shots), group in mean_df.groupby(["method", "shots_per_class"]):
        group = group.sort_values("epoch")
        style = {
            "color": colors[method],
            "linestyle": linestyles[shots],
            "linewidth": 2.0,
        }
        axes[0, 0].plot(group["epoch"], group["train_loss"], **style)
        axes[0, 1].plot(group["epoch"], group["val_loss"], **style)
        if has_train_acc:
            axes[1, 0].plot(group["epoch"], group["train_acc"], **style)
        axes[1, 1].plot(group["epoch"], group["val_acc"], **style)

    axes[0, 0].set_ylabel("Train loss")
    axes[0, 0].set_title("Training loss")

    axes[0, 1].set_ylabel("Source validation loss")
    axes[0, 1].set_title("Validation loss")

    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Train accuracy")
    axes[1, 0].set_title("Training accuracy")
    axes[1, 0].set_ylim(0, 1)

    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("Source validation accuracy")
    axes[1, 1].set_title("Validation accuracy")
    axes[1, 1].set_ylim(0, 1)

    for ax in axes.ravel():
        ax.grid(True, alpha=0.3)

    method_handles = [
        Line2D([0], [0], color=colors[method], lw=2, label=method)
        for method in methods
    ]
    shot_handles = [
        Line2D([0], [0], color="black", lw=2, linestyle=linestyles[shots],
               label=f"{shots} shots")
        for shots in shots_values
    ]
    method_legend = axes[1, 1].legend(
        handles=method_handles, title="Method", fontsize=8, title_fontsize=9,
        loc="lower right", handlelength=3.2,
    )
    axes[1, 1].add_artist(method_legend)
    axes[0, 0].legend(
        handles=shot_handles, title="Shots/class", fontsize=8, title_fontsize=9,
        loc="upper right", handlelength=4.0,
    )

    fig.tight_layout()
    return fig


## 8. Run Experiment Grid

This cell trains all configured methods over all shot counts and repeat seeds, then saves `results.csv`, `results.json`, `history.csv`, and `config.json`.

In [ ]:
set_seed(SEED)
device = get_device()

print("device:", device)
print("Dataset:", DATASET_NAME if DATASET_NAME != "medmnist" else f"medmnist/{MEDMNIST_NAME}")
print("Methods:", METHODS_TO_RUN)
print("Shots per class:", SHOTS_PER_CLASS)
print("Repeat seeds:", REPEAT_SEEDS)
print("Domain shift strength:", DOMAIN_SHIFT_STRENGTH)

backbone = HFViTBackbone().to(device)
freeze_module(backbone)
feature_dim = backbone.feature_dim
total_backbone_params = sum(p.numel() for p in backbone.parameters())

print(f"Backbone params: {total_backbone_params:,}")
print(f"Feature dim     : {feature_dim}")

data = load_experiment_data(backbone, device)
print(f"Classes         : {data['num_classes']}")
run_dir = make_output_dir()
print(f"Output dir      : {run_dir}")

results = []
for repeat_seed in REPEAT_SEEDS:
    for shots in SHOTS_PER_CLASS:
        for method in METHODS_TO_RUN:
            results.append(
                run_experiment(
                    method,
                    shots,
                    repeat_seed,
                    backbone,
                    feature_dim,
                    data,
                    device,
                )
            )

config_dict = experiment_config(device, data, feature_dim)
df = save_results(results, config_dict, run_dir)
print_results(df)


## 9. Plot Current Run

In [ ]:
df_plot, history_df, config_plot = load_run(run_dir)

fig = plot_accuracy_grid(df_plot, config_plot)
fig.savefig(run_dir / "accuracy_grid.png", dpi=160)
plt.show()

fig = plot_generalization_summary(df_plot)
fig.savefig(run_dir / "generalization_summary.png", dpi=160)
plt.show()

fig = plot_training_curves(history_df)
fig.savefig(run_dir / "training_curves.png", dpi=160)
plt.show()

fig = plot_budget_and_shift(df_plot)
fig.savefig(run_dir / "budget_and_shift.png", dpi=160)
plt.show()

print(f"Saved plots to {run_dir}")


## 10. Replot A Saved Run

Use this after restarting the notebook, or to inspect an older run directory.

In [ ]:
# Leave as None to use the latest run under outputs/part3.
RUN_DIR = None

selected_run_dir = Path(RUN_DIR).resolve() if RUN_DIR is not None else latest_run_dir()
df_plot, history_df, config_plot = load_run(selected_run_dir)
print("Run dir:", selected_run_dir)
print("Rows:", len(df_plot))

display(df_plot.head())

for filename, make_fig in [
    ("accuracy_grid.png", lambda: plot_accuracy_grid(df_plot, config_plot)),
    ("generalization_summary.png", lambda: plot_generalization_summary(df_plot)),
    ("training_curves.png", lambda: plot_training_curves(history_df)),
    ("budget_and_shift.png", lambda: plot_budget_and_shift(df_plot)),
]:
    fig = make_fig()
    fig.savefig(selected_run_dir / filename, dpi=160)
    plt.show()

print(f"Saved plots to {selected_run_dir}")
